In [1]:
# Cài đặt PySpark
%pip install pyspark

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.appName("MovieRatingsAnalysis").getOrCreate()

print("Spark Context đã được khởi tạo thành công!")

Spark Context đã được khởi tạo thành công!


In [3]:
# Đọc dữ liệu từ các file
import os


if 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    data_path = "/content/"
else:
    data_path = "data/"

# Đọc file movies.txt
movies_rdd = sc.textFile(data_path + "movies.txt")
print(f"Số lượng phim: {movies_rdd.count()}")

# Đọc file ratings_1.txt và ratings_2.txt
ratings_1_rdd = sc.textFile(data_path + "ratings_1.txt")
ratings_2_rdd = sc.textFile(data_path + "ratings_2.txt")

print(f"Số lượng rating từ file 1: {ratings_1_rdd.count()}")
print(f"Số lượng rating từ file 2: {ratings_2_rdd.count()}")

# Hiển thị dòng dữ liệu mẫu
print("\nDữ liệu movies.txt (5 dòng đầu):")
for line in movies_rdd.take(5):
    print(line)

print("\nDữ liệu ratings_1.txt (5 dòng đầu):")
for line in ratings_1_rdd.take(5):
    print(line)

Số lượng phim: 50
Số lượng rating từ file 1: 84
Số lượng rating từ file 2: 100

Dữ liệu movies.txt (5 dòng đầu):
1001,The Godfather (1972),Crime|Drama
1002,The Shawshank Redemption (1994),Drama
1003,Schindler's List (1993),Biography|Drama|History
1004,Raging Bull (1980),Biography|Drama|Sport
1005,Casablanca (1942),Drama|Romance|War

Dữ liệu ratings_1.txt (5 dòng đầu):
7,1020,4.5,1577836800
23,1015,3.5,1577923200
45,1030,4.0,1578009600
12,1047,3.0,1578096000
38,1012,4.5,1578182400


In [4]:
# Xử lý dữ liệu movies
# Parse movies.txt: MovieID, Title, Genres
def parse_movie(line):
    parts = line.split(',', 2)  # Tách thành 3 phần: ID, Title, Genres
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    return (movie_id, title)

movies_parsed = movies_rdd.map(parse_movie)
print("Movies parsed (5 records):")
for movie in movies_parsed.take(5):
    print(f"MovieID: {movie[0]}, Title: {movie[1]}")

# Tạo dictionary để tra cứu tên phim theo ID
movies_dict = movies_parsed.collectAsMap()
print(f"\nTổng số phim trong dictionary: {len(movies_dict)}")

Movies parsed (5 records):
MovieID: 1001, Title: The Godfather (1972)
MovieID: 1002, Title: The Shawshank Redemption (1994)
MovieID: 1003, Title: Schindler's List (1993)
MovieID: 1004, Title: Raging Bull (1980)
MovieID: 1005, Title: Casablanca (1942)

Tổng số phim trong dictionary: 50


In [5]:
# Xử lý dữ liệu ratings
# Parse ratings: UserID, MovieID, Rating, Timestamp
def parse_rating(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (movie_id, rating)

# Parse cả 2 file ratings
ratings_1_parsed = ratings_1_rdd.map(parse_rating)
ratings_2_parsed = ratings_2_rdd.map(parse_rating)

print("Ratings 1 parsed (5 records):")
for rating in ratings_1_parsed.take(5):
    print(f"MovieID: {rating[0]}, Rating: {rating[1]}")

print("\nRatings 2 parsed (5 records):")
for rating in ratings_2_parsed.take(5):
    print(f"MovieID: {rating[0]}, Rating: {rating[1]}")

# Gộp 2 RDD ratings lại
all_ratings = ratings_1_parsed.union(ratings_2_parsed)
print(f"\nTổng số ratings từ cả 2 file: {all_ratings.count()}")

Ratings 1 parsed (5 records):
MovieID: 1020, Rating: 4.5
MovieID: 1015, Rating: 3.5
MovieID: 1030, Rating: 4.0
MovieID: 1047, Rating: 3.0
MovieID: 1012, Rating: 4.5

Ratings 2 parsed (5 records):
MovieID: 1012, Rating: 3.5
MovieID: 1039, Rating: 4.0
MovieID: 1043, Rating: 4.5
MovieID: 1020, Rating: 3.0
MovieID: 1050, Rating: 4.0

Tổng số ratings từ cả 2 file: 184


In [6]:
# Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim

# Group theo MovieID và tính tổng rating + số lượng rating
# (movie_id, rating) -> (movie_id, (rating, 1))
ratings_with_count = all_ratings.map(lambda x: (x[0], (x[1], 1)))

print("Ratings with count (5 records):")
for record in ratings_with_count.take(5):
    print(f"MovieID: {record[0]}, (Rating: {record[1][0]}, Count: {record[1][1]})")

# Reduce theo key để tính tổng rating và tổng số lượng rating cho mỗi phim
# (movie_id, (sum_ratings, total_count))
movie_stats = ratings_with_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

print(f"\nTổng số phim có rating: {movie_stats.count()}")
print("Movie stats (5 records):")
for stat in movie_stats.take(5):
    print(f"MovieID: {stat[0]}, Sum: {stat[1][0]}, Count: {stat[1][1]}")

Ratings with count (5 records):
MovieID: 1020, (Rating: 4.5, Count: 1)
MovieID: 1015, (Rating: 3.5, Count: 1)
MovieID: 1030, (Rating: 4.0, Count: 1)
MovieID: 1047, (Rating: 3.0, Count: 1)
MovieID: 1012, (Rating: 4.5, Count: 1)

Tổng số phim có rating: 14
Movie stats (5 records):
MovieID: 1020, Sum: 66.0, Count: 18
MovieID: 1012, Sum: 8.0, Count: 2
MovieID: 1040, Sum: 65.0, Count: 18
MovieID: 1028, Sum: 24.5, Count: 7
MovieID: 1037, Sum: 70.0, Count: 18


In [7]:
# Tính điểm trung bình và thêm tên phim
# (movie_id, (average_rating, total_ratings, movie_title))
def calculate_average_and_add_title(record):
    movie_id, (sum_ratings, count) = record
    average_rating = sum_ratings / count
    movie_title = movies_dict.get(movie_id, f"Unknown Movie {movie_id}")
    return (movie_id, (average_rating, count, movie_title))

movie_results = movie_stats.map(calculate_average_and_add_title)

for result in movie_results.collect():
    movie_id, (avg_rating, count, title) = result
    print(f"{title} AverageRating: {avg_rating:.2f} (TotalRatings: {count})")

# Lọc những phim có ít nhất 5 lượt đánh giá
movies_with_min_ratings = movie_results.filter(lambda x: x[1][1] >= 5)

# Tìm phim có điểm trung bình cao nhất
# Đảm bảo RDD không rỗng trước khi gọi max
if movies_with_min_ratings.isEmpty():
    print("Không có phim nào có ít nhất 5 lượt đánh giá.")
else:
    highest_rated_movie = movies_with_min_ratings.max(key=lambda x: x[1][0])

    movie_id, (avg_rating, count, title) = highest_rated_movie
    print(f"\n{title} is the highest rated movie with an average rating of {avg_rating:.2f} among movies with at least 5 ratings.")


E.T. the Extra-Terrestrial (1982) AverageRating: 3.67 (TotalRatings: 18)
Psycho (1960) AverageRating: 4.00 (TotalRatings: 2)
Gladiator (2000) AverageRating: 3.61 (TotalRatings: 18)
Fight Club (1999) AverageRating: 3.50 (TotalRatings: 7)
The Lord of the Rings: The Fellowship of the Ring (2001) AverageRating: 3.89 (TotalRatings: 18)
The Terminator (1984) AverageRating: 4.06 (TotalRatings: 18)
The Godfather: Part II (1974) AverageRating: 4.00 (TotalRatings: 17)
The Silence of the Lambs (1991) AverageRating: 3.14 (TotalRatings: 7)
Mad Max: Fury Road (2015) AverageRating: 3.47 (TotalRatings: 18)
Lawrence of Arabia (1962) AverageRating: 3.44 (TotalRatings: 18)
Sunset Boulevard (1950) AverageRating: 4.36 (TotalRatings: 7)
The Social Network (2010) AverageRating: 3.86 (TotalRatings: 7)
No Country for Old Men (2007) AverageRating: 3.89 (TotalRatings: 18)
The Lord of the Rings: The Return of the King (2003) AverageRating: 3.82 (TotalRatings: 11)

Sunset Boulevard (1950) is the highest rated movi

In [8]:
# Dọn dẹp tài nguyên
sc.stop()
spark.stop()
print("Đã dừng Spark Context và Spark Session.")

Đã dừng Spark Context và Spark Session.
